# Shared memory: a moving-average stencil

For each output, this example averages a window of `2R+1` input values. Values outside the array are zero, and the denominator remains `2R+1` at the boundaries. The direct and shared-memory kernels compute the same formula.

## Select a GPU runtime

In Colab, choose **Runtime → Change runtime type → GPU**. This cell displays the selected device and creates a folder for the CUDA source files.

In [1]:
!nvidia-smi
!nvcc --version
!mkdir -p code


Sat Nov 14 06:57:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## CUDA error and timing support

The error-checking macro is independent of the timer. `GetCounter()` returns elapsed milliseconds and waits for the recorded work to finish.

In [2]:
%%writefile code/cuda_check.cuh
#ifndef CUDA_CHECK_CUH
#define CUDA_CHECK_CUH
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#define CUDA_CHECK(call) do { \
    cudaError_t status = (call); \
    if (status != cudaSuccess) { \
        std::fprintf(stderr, "%s:%d: %s\n", __FILE__, __LINE__, \
                     cudaGetErrorString(status)); \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)
#endif


Writing code/cuda_check.cuh


In [3]:
%%writefile code/TimingGPU.cuh
#ifndef TIMING_GPU_CUH
#define TIMING_GPU_CUH
#include <cuda_runtime.h>
class TimingGPU {
    cudaEvent_t start_;
    cudaEvent_t stop_;
    bool started_;
public:
    TimingGPU();
    ~TimingGPU();
    TimingGPU(const TimingGPU&) = delete;
    TimingGPU& operator=(const TimingGPU&) = delete;
    void StartCounter();
    float GetCounter(); // milliseconds
};
#endif


Writing code/TimingGPU.cuh


In [4]:
%%writefile code/TimingGPU.cu
#include "TimingGPU.cuh"
#include "cuda_check.cuh"
#include <cstdio>
#include <cstdlib>

TimingGPU::TimingGPU() : started_(false) {
    CUDA_CHECK(cudaEventCreate(&start_));
    CUDA_CHECK(cudaEventCreate(&stop_));
}

TimingGPU::~TimingGPU() {
    CUDA_CHECK(cudaEventDestroy(start_));
    CUDA_CHECK(cudaEventDestroy(stop_));
}

void TimingGPU::StartCounter() {
    CUDA_CHECK(cudaEventRecord(start_));
    started_ = true;
}

float TimingGPU::GetCounter() {
    if (!started_) {
        std::fprintf(stderr, "Call StartCounter before GetCounter.\n");
        std::exit(EXIT_FAILURE);
    }
    CUDA_CHECK(cudaEventRecord(stop_));
    CUDA_CHECK(cudaEventSynchronize(stop_));
    float milliseconds = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&milliseconds, start_, stop_));
    started_ = false;
    return milliseconds;
}


Writing code/TimingGPU.cu


## Kernels and host program

The direct and shared-memory kernels compute the same moving average. The host builds a CPU reference array, compares every GPU value with it, and prints the small example. The shared-memory kernel loads the tile and halos cooperatively; all threads reach `__syncthreads()`, including those with no valid output.

In [5]:
%%writefile code/chapter02_moving_average.cu
#include "cuda_check.cuh"
#include "TimingGPU.cuh"
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <cstring>

__global__ void moving_direct(const float *a, float *y, int n, int radius) {
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        float sum = 0.0f;
        for (int r = -radius; r <= radius; ++r) {
            const int k = i + r;
            if (k >= 0 && k < n) sum += a[k];
        }
        y[i] = sum / float(2 * radius + 1);
    }
}

__global__ void moving_shared(const float *a, float *y, int n, int radius) {
    extern __shared__ float tile[];
    const int t = threadIdx.x;
    const int first = blockIdx.x * blockDim.x;
    const int width = blockDim.x + 2 * radius;
    for (int q = t; q < width; q += blockDim.x) {
        const int k = first + q - radius;
        tile[q] = (k >= 0 && k < n) ? a[k] : 0.0f;
    }
    __syncthreads();
    const int i = first + t;
    if (i < n) {
        float sum = 0.0f;
        for (int r = -radius; r <= radius; ++r)
            sum += tile[t + radius + r];
        y[i] = sum / float(2 * radius + 1);
    }
}

void launch(int shared, const float *d_a, float *d_y,
            int n, int threads, int radius) {
    const int blocks = (n + threads - 1) / threads;
    if (shared) {
        const size_t bytes = size_t(threads + 2 * radius) * sizeof(float);
        moving_shared<<<blocks, threads, bytes>>>(d_a, d_y, n, radius);
    } else {
        moving_direct<<<blocks, threads>>>(d_a, d_y, n, radius);
    }
    CUDA_CHECK(cudaGetLastError());
}

void run_case(int n, int threads, int radius, bool show, int repeats) {
    const size_t bytes = size_t(n) * sizeof(float);
    float *h_a = (float*)std::malloc(bytes);
    float *h_cpu = (float*)std::malloc(bytes);
    float *h_direct = (float*)std::malloc(bytes);
    float *h_shared = (float*)std::malloc(bytes);
    float *d_a = nullptr, *d_direct = nullptr, *d_shared = nullptr;
    if (!h_a || !h_cpu || !h_direct || !h_shared) {
        std::fprintf(stderr, "Host allocation failed.\n");
        std::exit(EXIT_FAILURE);
    }
    for (int i = 0; i < n; ++i) h_a[i] = float((i % 101) - 50) / 10.0f;
    for (int i = 0; i < n; ++i) {
        float sum = 0.0f;
        for (int r = -radius; r <= radius; ++r) {
            const int k = i + r;
            if (k >= 0 && k < n) sum += h_a[k];
        }
        h_cpu[i] = sum / float(2 * radius + 1);
    }
    CUDA_CHECK(cudaMalloc((void**)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_direct, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_shared, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice));

    launch(0, d_a, d_direct, n, threads, radius);
    launch(1, d_a, d_shared, n, threads, radius);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(h_direct, d_direct, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_shared, d_shared, bytes, cudaMemcpyDeviceToHost));
    for (int i = 0; i < n; ++i) {
        if (std::fabs(h_direct[i] - h_cpu[i]) > 1.0e-5f) {
            std::fprintf(stderr, "CPU/direct-GPU mismatch at i=%d.\n", i);
            std::exit(EXIT_FAILURE);
        }
        if (std::fabs(h_shared[i] - h_cpu[i]) > 1.0e-5f) {
            std::fprintf(stderr, "CPU/shared-GPU mismatch at i=%d.\n", i);
            std::exit(EXIT_FAILURE);
        }
    }
    std::printf("CPU vs direct GPU: PASS (%d values)\n", n);
    std::printf("CPU vs shared GPU: PASS (%d values)\n", n);
    if (show) {
        std::printf(" i       a[i]        CPU    direct    shared\n");
        for (int i = 0; i < n; ++i)
            std::printf("%2d %10.3f %9.3f %9.3f %9.3f\n",
                        i, h_a[i], h_cpu[i], h_direct[i], h_shared[i]);
    } else {
        TimingGPU timer;
        float ms[2];
        for (int method = 0; method < 2; ++method) {
            for (int k = 0; k < 5; ++k)
                launch(method, d_a, method ? d_shared : d_direct,
                       n, threads, radius);
            CUDA_CHECK(cudaDeviceSynchronize());
            timer.StartCounter();
            for (int k = 0; k < repeats; ++k)
                launch(method, d_a, method ? d_shared : d_direct,
                       n, threads, radius);
            ms[method] = timer.GetCounter() / repeats;
        }
        std::printf("RESULT N=%d B=%d R=%d direct_ms=%.6f shared_ms=%.6f ratio=%.3f\n",
                    n, threads, radius, ms[0], ms[1], ms[0] / ms[1]);
    }
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_direct));
    CUDA_CHECK(cudaFree(d_shared));
    std::free(h_a); std::free(h_cpu); std::free(h_direct); std::free(h_shared);
}

int main(int argc, char **argv) {
    if (argc != 2 || (std::strcmp(argv[1], "--demo") != 0 &&
                      std::strcmp(argv[1], "--radius") != 0 &&
                      std::strcmp(argv[1], "--sizes") != 0)) {
        std::fprintf(stderr, "Usage: %s --demo | --radius | --sizes\n", argv[0]);
        return EXIT_FAILURE;
    }
    CUDA_CHECK(cudaSetDevice(0));
    if (std::strcmp(argv[1], "--demo") == 0) {
        run_case(10, 4, 2, true, 0);
    } else if (std::strcmp(argv[1], "--radius") == 0) {
        const int radii[] = {0, 1, 2, 4, 8};
        for (int radius : radii)
            run_case(1048576, 256, radius, false, 100);
    } else {
        const int sizes[] = {4099, 65539, 1048579};
        const int blocks[] = {128, 256};
        for (int n : sizes)
            for (int threads : blocks)
                run_case(n, threads, 4, false, 100);
    }
    return EXIT_SUCCESS;
}


Writing code/chapter02_moving_average.cu


## Compile

In [6]:
%%bash
set -eu
nvcc -O2 -std=c++14 -Icode code/chapter02_moving_average.cu code/TimingGPU.cu -o moving_average_demo


## Print and validate a small case

For N=10, B=4, and R=2, the program prints the CPU reference and both GPU outputs. It checks every element against the CPU result, including the first and last indices.

In [7]:
import subprocess

def run_and_show(command):
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='')
    result.check_returncode()

run_and_show(['./moving_average_demo', '--demo'])


CPU vs direct GPU: PASS (10 values)
CPU vs shared GPU: PASS (10 values)
 i       a[i]        CPU    direct    shared
 0     -5.000    -2.940    -2.940    -2.940
 1     -4.900    -3.880    -3.880    -3.880
 2     -4.800    -4.800    -4.800    -4.800
 3     -4.700    -4.700    -4.700    -4.700
 4     -4.600    -4.600    -4.600    -4.600
 5     -4.500    -4.500    -4.500    -4.500
 6     -4.400    -4.400    -4.400    -4.400
 7     -4.300    -4.300    -4.300    -4.300
 8     -4.200    -3.400    -3.400    -3.400
 9     -4.100    -2.520    -2.520    -2.520


## Increase the radius

For each radius, the program checks both GPU arrays against the CPU reference, then prints the two kernel times and their ratio. N=1,048,576 and B=256 stay fixed while R varies.

In [8]:
run_and_show(['./moving_average_demo', '--radius'])

CPU vs direct GPU: PASS (1048576 values)
CPU vs shared GPU: PASS (1048576 values)
RESULT N=1048576 B=256 R=0 direct_ms=0.037269 shared_ms=0.040182 ratio=0.928
CPU vs direct GPU: PASS (1048576 values)
CPU vs shared GPU: PASS (1048576 values)
RESULT N=1048576 B=256 R=1 direct_ms=0.042075 shared_ms=0.050257 ratio=0.837
CPU vs direct GPU: PASS (1048576 values)
CPU vs shared GPU: PASS (1048576 values)
RESULT N=1048576 B=256 R=2 direct_ms=0.040850 shared_ms=0.047800 ratio=0.855
CPU vs direct GPU: PASS (1048576 values)
CPU vs shared GPU: PASS (1048576 values)
RESULT N=1048576 B=256 R=4 direct_ms=0.043846 shared_ms=0.053146 ratio=0.825
CPU vs direct GPU: PASS (1048576 values)
CPU vs shared GPU: PASS (1048576 values)
RESULT N=1048576 B=256 R=8 direct_ms=0.053305 shared_ms=0.060477 ratio=0.881


## Change the vector length and block size

This comparison fixes R=4 and varies N and B. Each result line includes CPU/GPU validation, the direct and shared-memory timings, and their ratio.

In [9]:
run_and_show(['./moving_average_demo', '--sizes'])

CPU vs direct GPU: PASS (4099 values)
CPU vs shared GPU: PASS (4099 values)
RESULT N=4099 B=128 R=4 direct_ms=0.003541 shared_ms=0.003034 ratio=1.167
CPU vs direct GPU: PASS (4099 values)
CPU vs shared GPU: PASS (4099 values)
RESULT N=4099 B=256 R=4 direct_ms=0.003482 shared_ms=0.003321 ratio=1.048
CPU vs direct GPU: PASS (65539 values)
CPU vs shared GPU: PASS (65539 values)
RESULT N=65539 B=128 R=4 direct_ms=0.004278 shared_ms=0.004682 ratio=0.914
CPU vs direct GPU: PASS (65539 values)
CPU vs shared GPU: PASS (65539 values)
RESULT N=65539 B=256 R=4 direct_ms=0.004301 shared_ms=0.004814 ratio=0.894
CPU vs direct GPU: PASS (1048579 values)
CPU vs shared GPU: PASS (1048579 values)
RESULT N=1048579 B=128 R=4 direct_ms=0.043663 shared_ms=0.053263 ratio=0.820
CPU vs direct GPU: PASS (1048579 values)
CPU vs shared GPU: PASS (1048579 values)
RESULT N=1048579 B=256 R=4 direct_ms=0.044728 shared_ms=0.054167 ratio=0.826


## Reading the timing output

The ratio is `T_direct / T_shared`; values above 1 mean the shared-memory version was faster for that measured case. The timer excludes allocation, transfers, validation, and printing. Repeated launches may reuse cache data, so this is a kernel comparison rather than an application-level or DRAM-traffic measurement.